# buffer-copy_-inplace — faded example 3: Broadcast a per-column vector into a 2D buffer with copy_

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `buffer-copy_-inplace`. Running the beacon reports progress on the `PyTorch: in-place buffer copy` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: in-place buffer copy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`buffer-copy_-inplace`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "buffer-copy_-inplace"
DD_SUBTOPIC = "PyTorch: in-place buffer copy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`tensor.copy_(src)` broadcasts the source up to the destination's shape before writing in place. Copying a shape-(C,) row into a shape-(R, C) buffer fills every row with that vector, in place, preserving the buffer's identity. The right-aligned broadcast rule makes the (C,) source compatible with the (R, C) destination.

## Faded exercise 3

Implement `fill_rows_with(buf, row)` where `buf` has shape `(R, C)` and `row` has shape `(C,)`. Use a single in-place `copy_` that broadcasts `row` across all `R` rows of `buf`, preserving `id(buf)`. Complete the blanked copy step.

**Fill in:** Copies the (C,) row vector into the (R, C) buffer in place, broadcasting it across all rows.

In [ ]:
def fill_rows_with(buf: Tensor, row: Tensor) -> None:
    assert buf.dim() == 2 and row.shape == (buf.shape[1],)
    raise NotImplementedError()  # TODO: broadcast-copy row across all rows of buf in place


def _test():
    t.manual_seed(0)
    buf = t.zeros(3, 4)
    row = t.tensor([1.0, 2.0, 3.0, 4.0])
    before_id = id(buf)
    expected = row.unsqueeze(0).expand(3, 4).clone()
    ret = fill_rows_with(buf, row)
    assert ret is None, "function must return None (mutate in place)"
    assert id(buf) == before_id, "buffer identity must be preserved"
    assert buf.shape == (3, 4), "buffer shape must be unchanged"
    assert t.allclose(buf, expected), f"got {buf}, expected {expected}"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def fill_rows_with(buf: Tensor, row: Tensor) -> None:
    assert buf.dim() == 2 and row.shape == (buf.shape[1],)
    buf.copy_(row)
```
</details>